# Detecting and Correcting Spatial Bias in VGI Using Remote Sensing

**End-to-end reproducible pipeline** — from raw LiDAR point clouds to a statewide
map of OpenStreetMap (OSM) quality bias.

Volunteered Geographic Information (VGI) such as OpenStreetMap is highly accurate in
well-mapped urban areas but incomplete and stale in data-sparse regions. This notebook
reproduces the full analysis of the
[vgi-spatial-bias](https://github.com/rayford295/vgi-spatial-bias) project
(an [I-GUIDE Summer School 2026 project](https://i-guide.io/summer-school/summer-school-2026/summer-school-2026-projects/))
at its two working scales:

| Scale | Ground truth | Question answered |
|---|---|---|
| **Campus pilot** (2 × 2 km UIUC tile) | LiDAR + NAIP consensus | *Is OSM missing real objects, and where inside a city?* |
| **Statewide** (102 Illinois counties) | Census-normalized OSM internals | *Does OSM quality vary systematically with who lives there?* |

**Research questions** (full framing in
[`docs/PROJECT_DESCRIPTION.md`](https://github.com/rayford295/vgi-spatial-bias/blob/main/docs/PROJECT_DESCRIPTION.md)):

1. How does VGI accuracy and completeness vary across geographic and socioeconomic contexts?
2. Can remote sensing detect discrepancies in VGI data such as roads and buildings?
3. How can we develop scalable (AI) approaches to automatically improve VGI quality in data-sparse regions?

**Pipeline at a glance**

```
 LiDAR (USGS 3DEP) ──► Stage 1  classical detection      (DTM, buildings, trees)
                   └─► Stage 2  DGCNN semantic segmentation   [optional, GPU-friendly]
 NAIP (RGBN)       ──► Stage 3  optical land cover, LiDAR-fused paved layer
 OSM 2019/2026     ──► Stage 4  buildings: completeness, bias map, temporal validation
                   └─► Stage 5  roads: NAIP support, 2019→2026 evolution, major-roads subset
 OSM statewide + Census ─► Stage 6  102-county urban→rural bias gradient
```

**How to run.** Execute top-to-bottom. Every stage is a thin wrapper around a script in
[`src/`](https://github.com/rayford295/vgi-spatial-bias/tree/main/src) — the notebook
explains *what* each stage does and *why*, runs it, then displays the outputs. All
heavy inputs are downloaded automatically (≈ 1 GB on first run, cached afterwards).
Stage 2 (deep learning) is optional and controlled by the `RUN_DL` flag in the
configuration cell. Designed to run unmodified on the I-GUIDE JupyterHub, a laptop,
or any Linux/macOS machine with Python ≥ 3.10.

| Stage | Approx. runtime |
|---|---|
| Data download (first run only) | 5–15 min (network-bound) |
| 1 · Classical detection | ~3 min |
| 2 · DGCNN segmentation *(optional)* | 10–15 min on GPU/Apple-MPS, ~1–2 h on CPU |
| 3 · NAIP segmentation | ~2 min |
| 4 · Buildings comparison + temporal | ~3 min |
| 5 · Roads comparison + evolution | ~4 min |
| 6 · Statewide bias analysis | ~3 min |

## 1 · Environment setup

The notebook needs the **whole repository** (scripts, campus OSM subsets, docs), not
just this file. If you launched it from a repo checkout, nothing happens here; if you
only have the notebook (e.g. downloaded from an I-GUIDE knowledge element), the cell
clones the repository next to it and switches into it. It then installs the Python
dependencies pinned in `requirements.txt`.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/rayford295/vgi-spatial-bias.git"

if os.path.exists("src/prepare_data.py"):            # already inside a checkout
    REPO = os.getcwd()
else:                                                 # standalone notebook -> clone
    REPO = os.path.abspath("vgi-spatial-bias")
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    os.chdir(REPO)

print("repository:", REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=False)
print("dependencies ready")

### Configuration & helpers

`RUN_DL` toggles the deep-learning stage (everything else works without it — the
classical pipeline already provides the ground-truth footprints). The three helpers
keep every stage cell identical in shape: `run()` executes a pipeline script and fails
loudly if it fails, `show()` displays saved figures, `jshow()` pretty-prints a JSON
summary.

In [ ]:
%matplotlib inline
import json
from IPython.display import Image, display

RUN_DL = False          # set True to train the DGCNN segmentation model (GPU recommended)

def run(*args):
    cmd = [sys.executable] + list(args)
    print("$ python", " ".join(args))
    r = subprocess.run(cmd, cwd=REPO)
    if r.returncode != 0:
        raise RuntimeError(f"step failed: {' '.join(args)}")

def show(*paths, width=950):
    for p in paths:
        display(Image(filename=os.path.join(REPO, p), width=width))

def jshow(path, drop=()):
    with open(os.path.join(REPO, path)) as f:
        d = json.load(f)
    print(json.dumps({k: v for k, v in d.items() if k not in drop}, indent=2))

## 2 · Data acquisition

`src/prepare_data.py` downloads and stages every external input **idempotently**
(re-running skips files that are already present). All paths land where the pipeline
scripts expect them, and all locations are gitignored.

| Input | Source | Size | Staged at |
|---|---|---|---|
| LiDAR, 4 × 1 km QL1 tiles (80.8 M pts) | [I-GUIDE dataset storage](https://storage.i-guide.io) | ~340 MB | `UIUC_campus_LiDAR_merged_2x2km.laz` (tiles merged on the fly) |
| NAIP 4-band imagery, ~0.7 m, EPSG:6350 | GitHub release [`campus-rs-2019`](https://github.com/rayford295/vgi-spatial-bias/releases/tag/campus-rs-2019) | ~480 MB | `data/NAIP_image.tif` |
| OSM 2019 statewide major roads (375,754 segments, county-joined) | GitHub release [`osm-il-2019`](https://github.com/rayford295/vgi-spatial-bias/releases/tag/osm-il-2019) | ~65 MB | `data/statewide/OSM_2019_Major_Roads/` |
| Census 2019: county areas, population, boundaries | census.gov static files | ~5 MB | `data/statewide/` |

The small campus-extent OSM snapshots (`data/osm_*_20{19,26}.geojson`) are committed in
the repository, so the buildings/roads comparisons are reproducible byte-for-byte —
`src/fetch_osm_current.py` can refresh the "current" snapshot from the Overpass API if
you want to re-run the temporal validation against today's OSM instead of the archived
2026 snapshot.

**Why OSM 2019 and not current OSM as the subject?** The LiDAR and NAIP were flown in
2019. Comparing same-year data separates "OSM had not mapped it" from "it was not built
yet" — the 2026 snapshot is then used as *validation* (Stage 4) rather than as the
subject.

In [ ]:
run("src/prepare_data.py")

## 3 · Stage 1 — Classical LiDAR detection (the RS ground truth)

`src/classical_detection.py` turns the raw point cloud into objective, OSM-independent
reference layers:

1. **Ground / DTM** — morphological filtering of ground returns, then interpolation to a
   0.5 m bare-earth raster. Everything else is measured as *height above ground*.
2. **Building instances** — non-ground, low-vegetation-index, planar clusters →
   footprint polygons with `area_m2` and `height_m` attributes.
3. **Individual trees** — canopy-height-model local maxima → tree tops with crown size.

The ASPRS classification that ships inside the LAZ is used only as a cross-check; the
detection itself is reproducible from geometry alone. Outputs go to
`results/detection/` (GeoJSON in WGS84 for GIS hand-off + diagnostic PNGs).

> **Why LiDAR as ground truth?** Airborne LiDAR measures 3-D structure directly —
> a building is a planar elevated surface regardless of how it looks in imagery or
> whether any volunteer has ever mapped it. Fused with NAIP (Stage 3), two independent
> sensors must agree before we call an OSM omission real.

In [ ]:
run("src/classical_detection.py")

In [ ]:
show("results/detection/buildings_detected.png",
     "results/detection/trees_detected.png",
     "results/detection/ground_dtm.png")
jshow("results/detection/detection_summary.json")

## 4 · Stage 2 — Deep-learning semantic segmentation *(optional)*

`src/dgcnn_semseg.py` trains a **DGCNN (EdgeConv)** point-cloud segmentation model on
the ASPRS labels with a strict **spatial split** (west half = train, east half =
validation — random splits would leak neighboring points and inflate the score). The
key engineered feature is height-above-ground from Stage 1's DTM.

This stage demonstrates the *scalable-AI* direction of research question 3: a trained
model can label wall-to-wall tiles where no classified LiDAR exists. It is **not**
required for the bias analysis below (the classical footprints serve as ground truth),
so it is off by default — set `RUN_DL = True` in the configuration cell to train
(≈ 10–15 min on GPU/Apple-MPS; a CPU-only run can take 1–2 h).

Reference scores from the committed run: PointNet baseline OA 0.913 / mIoU 0.707 →
**DGCNN OA 0.930 / mIoU 0.768**.

In [ ]:
if RUN_DL:
    run("src/dgcnn_semseg.py")
else:
    print("RUN_DL = False -> skipping training; showing the committed reference figures")
show("results/segmentation/seg_fulltile.png", "results/segmentation/seg_confusion.png")
jshow("results/segmentation/seg_metrics.json")

## 5 · Stage 3 — NAIP optical segmentation (the corroborating sensor)

`src/naip_segmentation.py` derives land cover from the 4-band NAIP image:

- **Vegetation** — NDVI (NIR-based) OR excess-green index; robust and NAIP-only.
- **Impervious** — valid, non-vegetation, non-water pixels: the built-up extent.

Optical imagery has **no height information**, so buildings and pavement are
spectrally near-identical in a dense scene and cannot be separated by NAIP alone. The
resolution: intersect NAIP impervious with the LiDAR footprints —
`building = impervious ∩ LiDAR footprint`, and the remainder,
`paved = impervious − buildings`, becomes the road/parking reference used by Stage 5
(LiDAR has no road class, so roads need an optical reference). Crucially, NAIP's
impervious extent is **not trained on LiDAR** — the two sensors stay independent, which
is what makes their consensus meaningful.

In [ ]:
run("src/naip_segmentation.py", "data/NAIP_image.tif")

In [ ]:
show("results/naip/naip_segmentation.png")
jshow("results/naip/naip_summary.json")

## 6 · Stage 4 — Buildings: OSM 2019 vs the RS consensus

`src/vgi_comparison.py` evaluates the temporally-matched OSM 2019 `building=*` polygons
against the LiDAR footprints (corroborated by NAIP):

- **Matching** — IoU ≥ 0.3 between OSM and LiDAR footprints (EPSG:6350 metres),
  tracking 1:1 and 1:N cases.
- **Completeness** — matched / all LiDAR buildings, by count and by area. The two
  diverge when OSM maps the big buildings but skips the small ones.
- **Commission** — OSM polygons with no LiDAR support (stale, mislocated, or
  sub-threshold structures; interpreted per case, not automatically "wrong").
- **Gridded bias map** — completeness per 100 m cell: the *spatial* signature of bias.

`src/pixel_comparison.py` repeats the test at the pixel level (0.2 m), giving overall
accuracy, building-pixel IoU and Cohen's κ — a rasterized view that is insensitive to
how buildings are split into polygons.

In [ ]:
run("src/vgi_comparison.py", "data/osm_buildings_2019.geojson")
run("src/pixel_comparison.py", "data/osm_buildings_2019.geojson")

In [ ]:
show("results/comparison/comparison_map.png")
jshow("results/comparison/comparison_summary.json")
jshow("results/comparison/pixel/pixel_metrics_0p2m.json")

### Reading the result

OSM 2019 captures **58.3 % of buildings by count but 79.4 % by area** — the classic
VGI signature: large institutional buildings get mapped, small residential structures
(garages, sheds, back-lot houses) do not. The bias map shows completeness ≈ 1.0 on the
campus core collapsing **below 0.3 on the eastern residential strip** — a sharp spatial
gradient inside a single 2 × 2 km tile.

### Temporal validation — were the "omissions" real?

If the 2019 gaps were genuine unmapped buildings (not LiDAR artifacts or unbuilt lots),
the OSM community should have filled many of them since. Re-running the comparison
against the **OSM 2026** snapshot and diffing the two
(`src/temporal_validation.py`) classifies every 2019 gap as *filled by 2026* vs
*still unmapped*.

In [ ]:
run("src/vgi_comparison.py", "data/osm_buildings_2026.geojson", "osm2026")
run("src/temporal_validation.py")

In [ ]:
show("results/comparison/temporal/temporal_evolution.png")
jshow("results/comparison/temporal/temporal_summary.json")

**64 % of the 2019 gaps (352/547) were filled by the community by 2026** — they were in
the 2019 LiDAR all along, so they were real omissions, not yet-unbuilt structures.
Completeness rises to 81.9 % / 91.8 % against the 2026 snapshot. This validates the RS
consensus as an *early-warning* signal: remote sensing saw in 2019 what volunteers took
seven years to map.

## 7 · Stage 5 — Roads: OSM vs the NAIP paved layer

LiDAR has no road class, so the reference for roads is Stage 3's **paved layer**
(impervious − buildings). `src/road_comparison.py` tests both directions:

- **Forward (RS support)** — buffer each OSM way by a class-dependent width (+1.5 m
  positional tolerance) and measure the fraction of the buffer that is NAIP-paved.
  Support ≥ 0.5 ⇒ the way has physical pavement evidence.
- **Reverse (explained pavement)** — the fraction of paved pixels covered by any
  buffered OSM way. The remainder is *candidate* unmapped ways — but paved is a
  superset of roads (parking lots, plazas), so unexplained ≠ automatically missing.

Two follow-ups sharpen the picture:

- `src/road_evolution.py` classifies every way **added between 2019 and 2026** by
  whether it was already paved in the 2019 imagery — separating *gap-fill* (existed,
  unmapped) from *new construction*.
- `src/major_roads_analysis.py` runs the identical test on the vetted **major-roads
  subset** (`data/osm_roads_2019_major.geojson`, the campus clip of the statewide
  extract used in Stage 6) — the bridge between the two scales of this study.

Known caveats (documented in `results/comparison/README.md`): tree canopy shadows
optical pavement on residential streets (false "unsupported"), and legitimately unpaved
tracks/paths have no pavement to find.

In [ ]:
run("src/road_comparison.py")
run("src/road_evolution.py")
run("src/major_roads_analysis.py")

In [ ]:
show("results/comparison/roads/roads_2019.png",
     "results/comparison/temporal/roads_evolution.png",
     "results/comparison/roads/roads_2019_major.png")
jshow("results/comparison/roads/major_roads_summary.json", drop=("by_class",))

**Roads were already well-mapped in 2019 where buildings were not** — 91 % of way
length has pavement support, and the vetted major-roads subset is **99.6 % supported**
(the all-class shortfall is almost entirely canopy-shaded footways/steps). The
2019→2026 additions are dominated by micro-mapping (+60 % segments but only +8 %
length), and **80 % of the added length was already paved in 2019** — gap-fill again,
mirroring the buildings story. The major-roads clip also shows the attribute contrast
that motivates Stage 6: on campus, 26 % of ways carry a `maxspeed` and 94 % were edited
2017+, an order of magnitude above the statewide average.

## 8 · Stage 6 — Statewide scaling: the urban→rural bias gradient

The campus tile is one (extremely well-mapped) point in a much larger quality
landscape. `src/statewide_bias.py` scales the analysis to **375,754 major-road segments
(235,064 km) across all 102 Illinois counties**, asking research question 1 directly:
*does OSM quality vary systematically with who lives there?*

Per county it computes three families of metrics, then tests each against population
density (Spearman rank correlation — no linearity assumption) and contrasts urban
(≥ 100 persons/km²) vs rural counties:

- **Attribute completeness** — % of segments tagged with `maxspeed`, `surface`, `ref`,
  a street name. Geometry can exist while attributes are missing; attribute effort is a
  direct proxy for contributor attention.
- **Edit recency** — median `lastchange` year and % of segments touched 2017+. A road
  drawn once by a 2008 bulk import and never revisited is *unverified*, whatever its
  geometric quality.
- **Supply** — road-length density (km/km²) and km per 1,000 residents, normalized with
  Census 2019 land area and population estimates.

In [ ]:
run("src/statewide_bias.py",
    "data/statewide/OSM_2019_Major_Roads/gis_osm_roads_2019_IL_Major_Roads.shp",
    "data/statewide", "results/statewide")

In [ ]:
import pandas as pd
show("results/statewide/choropleth_completeness.png",
     "results/statewide/scatter_bias.png", width=1100)
pd.read_csv(os.path.join(REPO, "results/statewide/bias_tests.csv")).round(3)

### Reading the gradient

All correlations are significant at p < 0.001:

1. **Attribute completeness and recency are strongly urban-biased** (recency ρ = 0.70,
   maxspeed ρ = 0.50). Several downstate counties have a median last-edit year of
   **2008** — untouched since the original TIGER import — while Cook/DuPage/Will sit at
   2016 and the campus tile at 2018.
2. **Geometric supply is near-complete everywhere**: km per 1,000 residents is *higher*
   in rural counties (ρ = −0.97, an artifact of low population, not better mapping). In
   the US, the TIGER import means **bias lives in attributes and currency, not in
   whether the line exists** — which is exactly what the campus pilot found at object
   level.
3. **The gradient is not smooth.** Sangamon County (Springfield) tags `maxspeed` on
   33.9 % of segments — 3–10× any other county — the signature of a single dedicated
   contributor. Community attention, not just population, drives quality.

## 9 · Summary

| Finding | Evidence |
|---|---|
| OSM omissions are real and spatially structured | 58.3 % building completeness; < 0.3 on the residential strip (Stage 4) |
| RS consensus detects them years early | 64 % of 2019 gaps community-filled by 2026 (Stage 4) |
| Roads: geometry fine, attributes poor | 91–99.6 % pavement support (Stage 5) vs 3.3 % maxspeed statewide (Stage 6) |
| Quality follows contributors, not need | recency ρ = 0.70 with pop. density; TIGER-stale downstate; Sangamon hotspot (Stage 6) |

**Where this goes next (correction):** the detected omission polygons
(`results/comparison/omissions.geojson`) are machine-proposable OSM edits; the
statewide recency map prioritizes *where* automated calibration would pay off most; and
the Stage 2 model shows how labels can scale to tiles with no classified LiDAR.
Socioeconomic covariates (ACS income/education) are the natural extension of the
Stage 6 regression.

**Data credits** — USGS 3DEP LiDAR (public domain) · USDA NAIP (public domain) ·
© OpenStreetMap contributors (ODbL) · US Census Bureau. Code: MIT
([LICENSE](https://github.com/rayford295/vgi-spatial-bias/blob/main/LICENSE)).

*Project repository: <https://github.com/rayford295/vgi-spatial-bias> — an I-GUIDE
Summer School 2026 project.*